# Evaluate & Promote

Full evaluation of the stacked ensemble on the held-out test set, one
antisymmetric evidence observation per physical match.
The candidate is loaded from 04's manifest (never a latest-run lookup).

Promotion is a probability-first gate: the candidate replaces production
(`ensemble_lr_model` @champion) when its test log loss is strictly lower than
the incumbent's AND its ROC-AUC trails by no more than 0.01 on the same
`test_evidence` matrix. The four gate metrics (log_loss, roc_auc, accuracy,
brier) are computed for candidate and production and pinned for drift.
First promotion happens when no champion exists yet.

A report section always runs — ROC/PR/calibration curves, confusion matrices
and best-effort SHAP feature attribution — regardless of the promotion
outcome. Deployment (Bento build) is decoupled from promotion and out of this
path — see `src/flows/deploy.py` for the manual deploy flow.

In [ ]:
from src.utils import load_env

load_env()

from src.constants import (
    CANDIDATE_MANIFEST,
    CHAMPION_ALIAS,
    DATA_PROCESSED,
    PRODUCTION_MODEL,
)

input_dir = str(DATA_PROCESSED)
random_state = 42
production_model_name = PRODUCTION_MODEL  # MLflow registered model name
candidate_manifest = str(CANDIDATE_MANIFEST)  # written by 04
shap_sample_size = 1500  # SHAP subsample cap (report-only, wrapped in try/except)
force_promote = False  # --force-promote: bypass the metric gate and always promote
rebuild_cmd = ""  # optional manual deploy hook, off by default (never wired to a build)

In [ ]:
import hashlib
import json

import subprocess
from datetime import UTC, date, datetime
import numpy as np
import pandas as pd
import requests
import mlflow
from mlflow.tracking.client import MlflowClient


from src.constants import (
    BENTO_API_KEY,
    BENTO_API_KEY_HEADER,
    BATCH_MAX_SIZE_ROWS,
    EVAL_MAX_DATE_KEY,
    EVAL_SPLIT_SIZE_KEY,
    METRIC_PREFIX,
    MODEL_INFO_ROUTE,
    PREDICT_BATCH_ROUTE,
    PRODUCTION_BENTO_URL,
    PROMOTION_AUC_TOLERANCE,
    TRAIN_DATA_MAX_DATE_KEY,
    build_lineage_tags,
    FEATURE_COLS_HASH_TAG,
    FEATURE_COLS_TAG,
)

from src.evaluate.promotion import (
    METRIC_NAMES,
    check_candidate_cutoff,
    compute_metrics,
    decide_promotion,
    ordered_incumbent_contexts,
    resolve_champion,
    score_incumbent,
    verify_production_identity,
)

from src.evaluate.symmetry import evidence_to_probability

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

from sklearn.calibration import calibration_curve

import matplotlib.pyplot as plt

import seaborn as sns

In [ ]:
# ── Load test data (one antisymmetric evidence row per physical match) ──
# The stack head evaluates on the antisymmetric base evidence from 03
# (test_evidence.parquet + test_eval.parquet, one row per match) — never
# X_test or raw directional mirrors.
test_evidence = pd.read_parquet(f"{input_dir}/test_evidence.parquet")
test_eval = pd.read_parquet(f"{input_dir}/test_eval.parquet")
info_test = pd.read_parquet(f"{input_dir}/info_test.parquet").reset_index(drop=True)
y_test = pd.read_parquet(f"{input_dir}/y_test.parquet")["y"].reset_index(drop=True)

STACK_ORDER = ["linear", "gbdt", "nn"]
if list(test_evidence.columns) != STACK_ORDER:
    raise ValueError(
        f"test_evidence columns must be exactly {STACK_ORDER}, got {list(test_evidence.columns)}"
    )
if list(test_eval.columns) != ["match_id", "match_won"]:
    raise ValueError(
        f"test_eval columns must be [match_id, match_won], got {list(test_eval.columns)}"
    )
if not test_eval["match_id"].is_unique:
    raise ValueError("test_eval must hold exactly one row per physical match")
y_eval = test_eval["match_won"].reset_index(drop=True)

# One info_test row per physical match in the chosen orientation (03's
# deterministic convention: the row whose player_id is lexicographically
# smaller), ordered to match test_eval. Used for incumbent contexts and the
# surface error analysis below.
chosen_info = (
    info_test[
        np.asarray(info_test["player_id"], dtype=str)
        <= np.asarray(info_test["opponent_id"], dtype=str)
    ]
    .set_index("match_id")
    .loc[test_eval["match_id"]]
    .reset_index()
)
assert (chosen_info["match_id"] == test_eval["match_id"]).all()
print(f"Test set: {len(test_eval)} matches, evidence cols: {list(test_evidence.columns)}")

In [ ]:
# ── Load candidate from the manifest written by 04 ──
# Exact run handoff: candidate_run_id + model_uri from disk. No latest lookup.
with open(candidate_manifest) as f:
    manifest = json.load(f)
print(f"Candidate run:  {manifest['candidate_run_id']}")
print(f"Model URI:      {manifest['model_uri']}")
candidate = mlflow.sklearn.load_model(manifest["model_uri"])
assert candidate is not None
print("Loaded candidate meta-model")

In [ ]:
# ── Compute candidate predictions on the evidence matrix (one row per match) ──
assert candidate is not None
y_proba = candidate.predict_proba(test_evidence[STACK_ORDER].to_numpy())[:, 1]
y_pred = np.asarray(y_proba >= 0.5, dtype=int)

# Base-model comparison: each base's symmetric probability sigmoid(evidence),
# one observation per physical match.
print("Base model test ROC-AUC / Accuracy (symmetric evidence):")
for name in STACK_ORDER:
    p = evidence_to_probability(test_evidence[name])
    auc = roc_auc_score(y_eval, p)
    acc = accuracy_score(y_eval, np.asarray(p >= 0.5, dtype=int))
    print(f"  {name:8s} AUC {auc:.4f}  Acc {acc:.4f}")

In [ ]:
# ── Resolve the incumbent from the deployed production Bento ──

client = MlflowClient()

champion = resolve_champion(client)
prod_proba = None
prod_pred = None

if champion is not None:
    try:
        info_resp = requests.get(
            f"{PRODUCTION_BENTO_URL}{MODEL_INFO_ROUTE}",
            headers={BENTO_API_KEY_HEADER: BENTO_API_KEY},
            timeout=30,
        )
        info_resp.raise_for_status()
        verify_production_identity(info_resp.json(), champion)
        print(
            f"Production Bento verified: {production_model_name} "
            f"v{champion.version} run {champion.run_id}"
        )
        incumbent_contexts = ordered_incumbent_contexts(chosen_info)

        def post_incumbent_batch(chunk):
            resp = requests.post(
                f"{PRODUCTION_BENTO_URL}{PREDICT_BATCH_ROUTE}",
                headers={BENTO_API_KEY_HEADER: BENTO_API_KEY},
                json={"rows": chunk},
                timeout=120,
            )
            resp.raise_for_status()
            return resp.json()

        prod_proba = score_incumbent(
            incumbent_contexts, post_incumbent_batch, chunk_size=BATCH_MAX_SIZE_ROWS
        )
        prod_pred = (np.asarray(prod_proba) >= 0.5).astype(int)
        print(f"Scored incumbent through production Bento: {len(prod_proba)} rows")
    except (requests.RequestException, RuntimeError, TypeError) as exc:
        print(
            "Production Bento unavailable/incompatible"
            f" ({type(exc).__name__}: {exc}) — treating as first promotion."
        )
        champion = None

else:
    print("No production model found — first promotion.")

## Metrics — candidate vs production

All 4 gate metrics (log_loss, roc_auc, accuracy, brier) are computed for
candidate AND production on the same `test_evidence` matrix via the shared
`compute_metrics` contract.

In [ ]:
# ── Metrics ──

# Candidate-only headline numbers (kept for continuity).

print("Test Set Performance (candidate)")

print(f"ROC-AUC:    {roc_auc_score(y_eval, y_proba):.4f}")
print(f"Accuracy:   {accuracy_score(y_eval, y_pred):.4f}")
print(f"Precision:  {precision_score(y_eval, y_pred):.4f}")
print(f"Recall:     {recall_score(y_eval, y_pred):.4f}")
print(f"F1:         {f1_score(y_eval, y_pred):.4f}")

print()
print(classification_report(y_eval, y_pred))


# All 4 gate metrics for candidate AND incumbent via the shared promotion

# helpers (src.evaluate.promotion) on the same test_evidence matrix.

cand_metrics = compute_metrics(y_eval, y_proba, y_pred)
prod_metrics = compute_metrics(y_eval, prod_proba, prod_pred) if prod_proba is not None else None

print(f"\n{'metric':10s} {'candidate':>10s} {'production':>10s} {'delta':>10s}")

for m in METRIC_NAMES:
    if prod_metrics is not None:
        print(
            f"{m:10s} {cand_metrics[m]:10.4f} {prod_metrics[m]:10.4f} "
            f"{cand_metrics[m] - prod_metrics[m]:+10.4f}"
        )

    else:
        print(f"{m:10s} {cand_metrics[m]:10.4f} {'n/a':>10s} {'n/a':>10s}")

In [ ]:
# ── Error analysis by surface (chosen orientation, one observation per match) ──
if "surface" in info_test.columns:
    chosen_info["correct"] = (y_pred == y_eval).to_numpy()
    print("Accuracy by surface:")
    print(chosen_info.groupby("surface")["correct"].mean().to_string())

## Promotion decision

Probability-first gate: promote iff the candidate's test log loss is strictly
lower than the incumbent's AND its ROC-AUC trails by no more than
`PROMOTION_AUC_TOLERANCE` (0.01), or when no production exists yet (first
promotion). Idempotency guard kept: if `@champion` already points at this
run's version, no re-promotion.

In [ ]:
# ── Promotion decision: candidate vs incumbent on the SAME test_evidence ──

# Candidate data cutoff: reject before registration when any split's data
# extends beyond the current UTC date.

from src.features.columns import FEATURE_COLS

with open(f"{input_dir}/split_meta.json") as f:
    split_meta = json.load(f)

check_candidate_cutoff(date.fromisoformat(split_meta["max_match_date"]), datetime.now(UTC).date())

feature_cols = [str(col) for col in FEATURE_COLS]
feature_cols_json = json.dumps(feature_cols, separators=(",", ":"))
feature_cols_hash = hashlib.sha256(feature_cols_json.encode()).hexdigest()
print(f"Feature contract: {len(feature_cols)} columns ({feature_cols_hash[:12]})")


# Probability-first gate: strictly lower test log loss and ROC-AUC no more
# than PROMOTION_AUC_TOLERANCE below the incumbent's. Printed here as
# diagnostics; decide_promotion below makes the call.

if prod_metrics is not None:
    print("Promotion gate diagnostics (candidate vs production):")

    print(
        f"  log_loss  {cand_metrics['log_loss']:.4f} vs {prod_metrics['log_loss']:.4f} "
        f"(must be strictly lower to promote)"
    )

    print(
        f"  roc_auc   {cand_metrics['roc_auc']:.4f} vs {prod_metrics['roc_auc']:.4f} "
        f"(may trail by at most {PROMOTION_AUC_TOLERANCE})"
    )

else:
    print("No production model found — first promotion.")


# Idempotency guard kept: no re-promotion when @champion already points at this run.

champion_run_id = getattr(champion, "run_id", None)

promoted = decide_promotion(
    cand_metrics=cand_metrics,
    prod_metrics=prod_metrics,
    champion_run_id=champion_run_id,
    candidate_run_id=manifest["candidate_run_id"],
    force=force_promote,
)
print(
    f"Promotion decision: {'PROMOTE' if promoted else 'SKIP'}; candidate run={manifest['candidate_run_id']}"
)


if force_promote:
    print(">>> FORCE-PROMOTE: bypassing the metric gate and registering regardless")

elif champion_run_id is not None and str(champion_run_id) == str(manifest["candidate_run_id"]):
    print(">>> SKIP: production already points at this candidate run")

elif promoted:
    print(
        ">>> PROMOTE: candidate log loss improves and ROC-AUC is within tolerance (or first promotion)"
    )

else:
    print(">>> SKIP: production is still better")

In [ ]:
# ── Log decision; promotion registers only, deployment is manual ──
mlflow.set_experiment("evaluation")
with mlflow.start_run():
    mlflow.log_metrics({f"candidate_{m}": float(cand_metrics[m]) for m in METRIC_NAMES})
    if prod_metrics is not None:
        mlflow.log_metrics({f"production_{m}": float(prod_metrics[m]) for m in METRIC_NAMES})
        mlflow.log_metrics(
            {f"delta_{m}": float(cand_metrics[m] - prod_metrics[m]) for m in METRIC_NAMES}
        )
    mlflow.log_metric("promoted", promoted)
    mlflow.log_param("candidate_run_id", manifest["candidate_run_id"])

    eval_run = mlflow.active_run()
    assert eval_run is not None
    eval_run_id = eval_run.info.run_id

    if promoted:
        # Register every model together only on promotion: the ensemble plus
        # all three bases (02 logged them as runs but never registered).
        registered_pins = {}
        for name, pin in manifest["base_pins"].items():
            base_mv = mlflow.register_model(pin["model_uri"], pin["registered_model_name"])
            registered_pins[name] = {
                **pin,
                "version": str(base_mv.version),
                "model_uri": f"models:/{pin['registered_model_name']}/{base_mv.version}",
            }
            print(f"Registered base {name} as {pin['registered_model_name']} v{base_mv.version}")

        mv = mlflow.register_model(manifest["model_uri"], production_model_name)
        # Freeze exact lineage on the promoted version BEFORE @champion is
        # assigned: these tags are the only authority deploy resolves from.
        lineage_tags = build_lineage_tags(registered_pins, manifest["aux_pins"])
        lineage_tags[FEATURE_COLS_TAG] = feature_cols_json
        lineage_tags[FEATURE_COLS_HASH_TAG] = feature_cols_hash
        for key, value in lineage_tags.items():
            client.set_model_version_tag(production_model_name, str(mv.version), key, value)
        # Pin the training-data watermark (latest match date present in the
        # training splits) so drift checks cut off on data, not registration time.
        client.set_model_version_tag(
            production_model_name,
            str(mv.version),
            TRAIN_DATA_MAX_DATE_KEY,
            split_meta["max_match_date"],
        )
        # Pin the champion's 4 gate metrics so drift compares current
        # performance against this promotion-time reference.
        for m in METRIC_NAMES:
            client.set_model_version_tag(
                production_model_name,
                str(mv.version),
                f"{METRIC_PREFIX}{m}",
                str(float(cand_metrics[m])),
            )
        eval_tags = {
            EVAL_SPLIT_SIZE_KEY: str(len(test_eval)),
            EVAL_MAX_DATE_KEY: str(pd.to_datetime(chosen_info["match_date"]).max().date()),
        }
        for key, value in eval_tags.items():
            client.set_model_version_tag(production_model_name, str(mv.version), key, value)
        client.set_registered_model_alias(production_model_name, CHAMPION_ALIAS, str(mv.version))
        champion_after = client.get_model_version_by_alias(production_model_name, CHAMPION_ALIAS)
        print(
            f"Champion updated: {production_model_name} v{champion_after.version} "
            f"(run {champion_after.run_id}, alias @{CHAMPION_ALIAS})"
        )
        print(
            f"Registered {manifest['model_uri']} as '{production_model_name}' v{mv.version} "
            f"(@{CHAMPION_ALIAS}) with {len(lineage_tags)} lineage tags"
        )
        # Optional manual deploy hook — off by default, never wired to a build.
        if rebuild_cmd:
            print(f"Executing manual deploy hook: {rebuild_cmd}")
            result = subprocess.run(rebuild_cmd, shell=True)
            print(f"Deploy hook finished with return code {result.returncode}")
    else:
        print("No promotion — skipped registration. Deployment (if wanted) stays manual.")
    if not promoted:
        current = resolve_champion(client)
        print(
            f"Champion unchanged: {production_model_name} "
            f"v{getattr(current, 'version', 'none')} (candidate was not registered)"
        )
    print(f"Evaluation logged. promoted={promoted}")

## Report — always runs

Everything below renders unconditionally (the promotion outcome does not gate
it) and is logged to the evaluation run: ROC curves for the three base models
plus candidate and production, PR and calibration curves for candidate vs
production, side-by-side confusion matrices, and (best-effort) SHAP feature
attribution on the pinned GBDT candidate model.

In [ ]:
# ── ROC curves: 3 base models + candidate + production ──
with mlflow.start_run(run_id=eval_run_id):
    fig, ax = plt.subplots(figsize=(8, 6))
    for name in STACK_ORDER:
        p = evidence_to_probability(test_evidence[name])
        fpr, tpr, _ = roc_curve(y_eval, p)
        ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_eval, p):.3f})", lw=1.5)
    fpr, tpr, _ = roc_curve(y_eval, y_proba)
    ax.plot(
        fpr, tpr, label=f"candidate (AUC {roc_auc_score(y_eval, y_proba):.3f})", lw=2.5, ls="--"
    )
    if prod_proba is not None:
        fpr, tpr, _ = roc_curve(y_eval, prod_proba)
        ax.plot(
            fpr,
            tpr,
            label=f"production (AUC {roc_auc_score(y_eval, prod_proba):.3f})",
            lw=2.5,
            ls=":",
        )
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("ROC curves — base models, candidate, production")
    ax.legend(loc="lower right")
    fig.tight_layout()
    mlflow.log_figure(fig, "figures/roc_curves.png")
    plt.close(fig)

In [ ]:
# ── PR curves: candidate + production ──
with mlflow.start_run(run_id=eval_run_id):
    fig, ax = plt.subplots(figsize=(8, 6))
    precision, recall, _ = precision_recall_curve(y_eval, y_proba)
    ax.plot(
        recall,
        precision,
        label=f"candidate (AP {average_precision_score(y_eval, y_proba):.3f})",
        lw=2.5,
    )
    if prod_proba is not None:
        precision, recall, _ = precision_recall_curve(y_eval, prod_proba)
        ax.plot(
            recall,
            precision,
            label=f"production (AP {average_precision_score(y_eval, prod_proba):.3f})",
            lw=2.5,
            ls="--",
        )
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-recall curves — candidate vs production")
    ax.legend(loc="lower left")
    fig.tight_layout()
    mlflow.log_figure(fig, "figures/pr_curves.png")
    plt.close(fig)

In [ ]:
# ── Calibration (reliability) curves: candidate + production ──
with mlflow.start_run(run_id=eval_run_id):
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="perfect")
    prob_true, prob_pred = calibration_curve(y_eval, y_proba, n_bins=10)
    ax.plot(
        prob_pred,
        prob_true,
        marker="o",
        label=f"candidate (Brier {brier_score_loss(y_eval, y_proba):.4f})",
    )
    if prod_proba is not None:
        prob_true, prob_pred = calibration_curve(y_eval, prod_proba, n_bins=10)
        ax.plot(
            prob_pred,
            prob_true,
            marker="s",
            ls="--",
            label=f"production (Brier {brier_score_loss(y_eval, prod_proba):.4f})",
        )
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_title("Calibration curves — candidate vs production")
    ax.legend(loc="lower right")
    fig.tight_layout()
    mlflow.log_figure(fig, "figures/calibration_curves.png")
    plt.close(fig)

In [ ]:
# ── Confusion matrices: candidate + production side by side ──
with mlflow.start_run(run_id=eval_run_id):
    models_to_plot = [("candidate", y_pred)]
    if prod_pred is not None:
        models_to_plot.append(("production", prod_pred))
    fig, axes = plt.subplots(1, len(models_to_plot), figsize=(6 * len(models_to_plot), 5))
    if len(models_to_plot) == 1:
        axes = [axes]
    for ax, (title, pred) in zip(axes, models_to_plot, strict=False):
        sns.heatmap(confusion_matrix(y_eval, pred), annot=True, fmt="d", cmap="Blues", ax=ax)
        ax.set_title(f"{title} confusion matrix")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
    fig.tight_layout()
    mlflow.log_figure(fig, "figures/confusion_matrices.png")
    plt.close(fig)

## SHAP feature attribution (best-effort)

`shap.TreeExplainer` on the pinned GBDT candidate model (loaded from MLflow,
never a latest lookup), applied to a subsample of `X_test` limited to the
`FEATURE_COLS` column list. Beeswarm summary plus waterfall plots on
misclassified examples. Wrapped in try/except — a GBDT flavor shap cannot
handle only skips this section; the rest of the report still renders.

SHAP runs on the raw directional test rows (`X_test`, two rows per match),
not the antisymmetric evidence — attribution is per model input row, so this
section is unchanged by the evidence conversion.

In [ ]:
# ── SHAP: TreeExplainer on the pinned GBDT candidate (best-effort, never blocks the report) ──
# Slow/fragile on some GBDT packages — any failure logs a warning and the
# rest of the report still renders.
try:
    import shap

    from src.features.columns import FEATURE_COLS as feature_cols

    X_shap = pd.read_parquet(f"{input_dir}/X_test.parquet").reset_index(drop=True)
    cols = [c for c in feature_cols if c in X_shap.columns]
    X_shap = X_shap[cols]
    sample = X_shap.sample(n=min(shap_sample_size, len(X_shap)), random_state=random_state)
    y_sample = y_test.iloc[sample.index].reset_index(drop=True)

    gbdt = mlflow.sklearn.load_model(manifest["base_pins"]["gbdt"]["model_uri"])
    assert gbdt is not None
    explainer = shap.TreeExplainer(gbdt)
    shap_values = explainer(sample)
    print(
        f"SHAP computed on {len(sample)} rows x {len(cols)} features "
        f"({len(feature_cols)} declared in FEATURE_COLS)"
    )

    with mlflow.start_run(run_id=eval_run_id):
        shap.plots.beeswarm(shap_values, show=False)
        plt.title("SHAP beeswarm — pinned gbdt")
        mlflow.log_figure(plt.gcf(), "figures/shap_beeswarm.png")
        plt.close(plt.gcf())

        mis_pos = np.where(gbdt.predict(sample) != y_sample.to_numpy())[0]
        for i, pos in enumerate(mis_pos[:3]):
            shap.plots.waterfall(shap_values[pos], show=False)
            plt.title(f"SHAP waterfall — misclassified sample (row {sample.index[pos]})")
            mlflow.log_figure(plt.gcf(), f"figures/shap_waterfall_misclassified_{i}.png")
            plt.close(plt.gcf())
        print(f"SHAP waterfall for {min(len(mis_pos), 3)} misclassified examples")
except Exception as e:
    print(f"WARNING: SHAP analysis failed and was skipped: {e}")